## Airline AI Assistant - Tools Calling

In [17]:
# libraries
from dotenv import load_dotenv
import os, json
from openai import OpenAI
import gradio as gr
import sqlite3

In [2]:
# set up environment
MODEL_GPT = 'gpt-4.1-mini'
MODEL_LLAMA = 'llama3.2'

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print("API Keys not found")
elif not api_key.startswith('sk-proj-'):
    print("API key found but wrong format")
elif api_key.strip() != api_key:
    print("API key found but contain unnecessary space")
else:
    print("API Key found and in use")


openai = OpenAI()

OLLAMA_BASE_URL = 'http://localhost:11434/v1'
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

API Key found and in use


In [3]:
system_message = """
    You are a helpful assistant for an Airline called FlightAI.
    Give short, courteous answers, no more than 1 sentence.
    Always be accurate. If you don't know the answer, say so.
"""

In [4]:
def chat(message, history):
    history = [{'role': h["role"], 'content': h["content"]} for h in history]
    messages = (
        [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    )

    response = openai.chat.completions.create(model=MODEL_GPT, messages=messages)

    return response.choices[0].message.content


gr.ChatInterface(fn=chat, type='messages').launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


### Tools

In [14]:
ticket_prices = {'London': '$799', 'Paris': '$999', 'Tokyo': '$1400', 'Italy': '$1200'}

def get_ticket_price(destination_city):
    print(f"Tool called for city {destination_city}")

    price = ticket_prices.get(destination_city.capitalize(), 'Unknown City')

    return f"The price of a ticket to {destination_city.capitalize()} is {price}"

get_ticket_price('italy')

Tool called for city italy


'The price of a ticket to Italy is $1200'

In [6]:
# There's a particular dictionary structure that's required to describe our function:

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

In [7]:
tools = [{"type": 'function', 'function': price_function }]
tools

[{'type': 'function',
  'function': {'name': 'get_ticket_price',
   'description': 'Get the price of a return ticket to the destination city.',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'The city that the customer wants to travel to'}},
    'required': ['destination_city'],
    'additionalProperties': False}}}]

In [8]:
# We have to write that function handle_tool_call:

def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    if tool_call.function.name == "get_ticket_price":
        arguments = json.loads(tool_call.function.arguments)
        city = arguments.get('destination_city')
        price_details = get_ticket_price(city)
        response = {
            "role": "tool",
            "content": price_details,
            "tool_call_id": tool_call.id
        }
    return response

In [9]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL_GPT, messages=messages, tools=tools)

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        response = handle_tool_call(message)
        messages.append(message)
        messages.append(response)
        response = openai.chat.completions.create(model=MODEL_GPT, messages=messages)
    
    return response.choices[0].message.content

In [10]:
gr.ChatInterface(fn=chat, type='messages').launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


In [11]:
# Improvement to handle multiple tool calls
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL_GPT, messages=messages, tools=tools)

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL_GPT, messages=messages)
    
    return response.choices[0].message.content

In [12]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses

In [15]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


Tool called for city London
Tool called for city Italy


In [16]:
# Improvement to handle multiple tool calls
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL_GPT, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL_GPT, messages=messages, tools=tools)
    
    return response.choices[0].message.content

### Tool Calling with Database - Sqlite

In [18]:
DB = 'prices.db'

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()

    cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
    conn.commit()

In [19]:
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)

    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))

        result = cursor.fetchone()

        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

In [20]:
get_ticket_price('London')

DATABASE TOOL CALLED: Getting price for London


'No price data available for this city'

In [22]:
def set_ticket_price(city, price):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()


for city, price in ticket_prices.items():
    set_ticket_price(city, price)

get_ticket_price('London')

DATABASE TOOL CALLED: Getting price for London


'Ticket price to London is $$799'

In [23]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


DATABASE TOOL CALLED: Getting price for Italy
